In [1]:
import os
from pathlib import Path

DATA_ROOT = Path.home() / "mne_data"

os.environ["MNE_DATA"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_BNCI_PATH"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_ZHOU2016_PATH"] = str(DATA_ROOT)
os.environ["MOABB_DOWNLOAD_PROVIDER"] = "upstream"

EXPERIMENT_DIR = Path("/Users/liza/Projects/vkr/mi-ood")
CACHE_DIR = EXPERIMENT_DIR / "cache"
RESULTS_DIR = EXPERIMENT_DIR / "results"

print(EXPERIMENT_DIR)

/Users/liza/Projects/vkr/mi-ood


In [2]:
from methods import ShallowOOD
from pipelines import mi_pipelines, ood_pipelines
from processing import *

/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/optuna/integration/sklearn.py:14: FutureWarning: `optuna.integration.sklearn` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.sklearn` instead.
  optuna_warn(f"{msg} Use `optuna_integration.sklearn` instead.", FutureWarning)
/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/braindecode/models/eegpt.py:497: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage("standard_1020")
/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/braindecode/models/eegpt.py:1452: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = make_standard_montage("standard_1020")


In [3]:
experiment = "within-session_7-channels"
COMMON_CHANNELS = ["FC3", "FCz", "FC4", "C3", "C4", "CP3", "CP4"]
datasets = [ "TTK", "Zhou2016", "BNCI2014_001", "Wang2025", "Yang2025_3C", "Zhou2020" ]


In [4]:
for dataset in datasets:
    print( dataset )
    subjects = get_subjects(dataset) 

    for subject in subjects:
        data = None
        for method, pipeline in mi_pipelines.items():
            result = {"dataset": dataset, "subject": subject, "experiment": experiment, "experiment_type": "mi", "method": method}
            if result_path(result).exists():
                continue
            if data is None:
                data = load_subject_data(dataset, subject, channels=COMMON_CHANNELS, within_session=True)
            train_data, val_data, test_data = data
            result["MI_BA"], result["MI_AUROC"] = MI_metrics(train_data, test_data, pipeline)
            save_result(result)
            print(f"{dataset} s{subject} {method}: {result['MI_BA']:.3f} {result['MI_AUROC']:.3f}")

        data = None
        for method, pipeline in ood_pipelines.items():
            result = {"dataset": dataset, "subject": subject, "experiment": experiment, "experiment_type": "ood", "method": method}
            if result_path(result).exists():
                continue
            if data is None:
                data = load_subject_data(dataset, subject, channels=COMMON_CHANNELS, ood_classes=("feet",), within_session=True)
                train_data, val_data, test_data = data
            model = train(train_data, pipeline)

            result["OOD_AUROC"] = OOD_AUROC(test_data, model)
            save_result(result)
            print(f"{dataset} s{subject} {method}: {result['OOD_AUROC']:.3f}")

    display_summary_by_subject(dataset, experiment, experiment_type="mi", metric="MI_BA")
    display_summary_by_subject(dataset, experiment, experiment_type="mi", metric="MI_AUROC")
    display_summary_by_subject(dataset, experiment, experiment_type="ood", metric="OOD_AUROC")

TTK


,MI_BA,n,sub-01,sub-03,sub-04,sub-05,sub-06,sub-07
method,,,,,,,,
ACM+TS+SVM,0.735,6,0.800,0.715,0.760,0.558,0.938,0.641
FBCSP+ElasticNet,0.721,6,0.767,0.878,0.510,0.561,0.929,0.683
Log-bandpower+shLDA,0.700,6,0.833,0.519,0.715,0.596,0.866,0.673
Microstates+GB,0.646,6,0.867,0.519,0.554,0.442,0.938,0.554
EEGNet,0.633,6,0.900,0.599,0.558,0.401,0.857,0.481
VAR+shLDA,0.622,6,0.733,0.638,0.551,0.442,0.804,0.561
ShallowConvNet,0.604,6,0.833,0.676,0.478,0.439,0.804,0.394


,MI_AUROC,n,sub-01,sub-03,sub-04,sub-05,sub-06,sub-07
method,,,,,,,,
FBCSP+ElasticNet,0.804,6,0.889,0.962,0.628,0.577,1.000,0.769
ACM+TS+SVM,0.773,6,0.867,0.724,0.801,0.590,0.982,0.673
Log-bandpower+shLDA,0.750,6,0.876,0.500,0.821,0.628,0.946,0.731
EEGNet,0.658,6,0.991,0.571,0.545,0.417,0.964,0.462
VAR+shLDA,0.648,6,0.867,0.590,0.603,0.481,0.821,0.526
ShallowConvNet,0.646,6,0.960,0.692,0.487,0.513,0.768,0.455
Microstates+GB,0.638,6,0.933,0.564,0.487,0.397,0.875,0.571


,OOD_AUROC,n,sub-01,sub-03,sub-04,sub-05,sub-06,sub-07
method,,,,,,,,
ShallowConvNet+kNN,0.762,6,0.733,0.775,0.607,0.812,0.890,0.753
FBCSP+IsolationForest,0.761,6,0.633,0.918,0.588,0.664,0.932,0.827
EEGNet+TempDens,0.748,6,0.802,0.800,0.536,0.596,0.929,0.822
ACM+TS+OCSVM,0.688,6,0.520,0.757,0.596,0.628,0.791,0.837
VAR+Mahalanobis,0.661,6,0.676,0.888,0.485,0.567,0.606,0.742
Log-bandpower+Mahalanobis,0.654,6,0.654,0.882,0.525,0.626,0.656,0.581
Microstates+LOF,0.583,6,0.615,0.502,0.520,0.636,0.667,0.559


Zhou2016


,MI_BA,n,sub-01,sub-02,sub-03,sub-04
method,,,,,,
FBCSP+ElasticNet,0.894,4,0.867,0.756,1.000,0.955
EEGNet,0.870,4,0.900,0.715,1.000,0.864
ACM+TS+SVM,0.852,4,0.733,0.756,0.958,0.958
Microstates+GB,0.841,4,0.767,0.718,0.920,0.958
Log-bandpower+shLDA,0.825,4,0.767,0.753,0.958,0.822
ShallowConvNet,0.773,4,0.900,0.718,0.917,0.557
VAR+shLDA,0.685,4,0.733,0.635,0.753,0.617


... (14 outputs left out)

Zhou2020


,MI_BA,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20
method,,,,,,,,,,,,,,,,,,,,,,
ACM+TS+SVM,0.608,20,0.633,0.400,0.700,0.533,0.600,0.833,0.933,0.433,0.467,0.400,0.733,0.633,0.533,0.667,0.833,0.467,0.459,0.767,0.400,0.733
FBCSP+ElasticNet,0.602,20,0.500,0.467,0.567,0.533,0.533,0.800,0.867,0.500,0.733,0.567,0.600,0.533,0.467,0.667,0.800,0.667,0.515,0.667,0.533,0.533
EEGNet,0.602,20,0.567,0.633,0.700,0.400,0.700,0.867,0.933,0.500,0.500,0.467,0.700,0.667,0.367,0.633,0.700,0.500,0.409,0.633,0.400,0.767
ShallowConvNet,0.598,20,0.533,0.633,0.667,0.367,0.467,0.867,0.833,0.533,0.533,0.467,0.733,0.667,0.567,0.533,0.733,0.467,0.459,0.700,0.400,0.800
Log-bandpower+shLDA,0.574,20,0.500,0.467,0.633,0.367,0.633,0.700,0.700,0.700,0.467,0.533,0.500,0.567,0.533,0.633,0.767,0.533,0.515,0.633,0.500,0.600
Microstates+GB,0.542,20,0.600,0.467,0.633,0.333,0.600,0.733,0.867,0.467,0.467,0.433,0.467,0.433,0.400,0.533,0.633,0.467,0.512,0.700,0.533,0.567
VAR+shLDA,0.540,20,0.433,0.500,0.567,0.533,0.467,0.733,0.700,0.500,0.633,0.367,0.500,0.533,0.400,0.700,0.767,0.500,0.408,0.567,0.333,0.667


,MI_AUROC,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20
method,,,,,,,,,,,,,,,,,,,,,,
ACM+TS+SVM,0.644,20,0.649,0.342,0.756,0.636,0.649,0.938,0.978,0.440,0.520,0.378,0.742,0.640,0.431,0.769,0.911,0.556,0.515,0.791,0.458,0.787
ShallowConvNet,0.634,20,0.458,0.618,0.760,0.382,0.609,0.907,0.924,0.498,0.596,0.471,0.773,0.582,0.600,0.578,0.858,0.640,0.488,0.760,0.418,0.769
FBCSP+ElasticNet,0.630,20,0.480,0.502,0.671,0.640,0.587,0.871,0.960,0.569,0.733,0.458,0.644,0.604,0.364,0.716,0.902,0.653,0.500,0.724,0.440,0.582
EEGNet,0.622,20,0.493,0.636,0.787,0.444,0.693,0.956,0.982,0.600,0.484,0.289,0.818,0.689,0.391,0.702,0.840,0.422,0.412,0.702,0.378,0.711
Log-bandpower+shLDA,0.590,20,0.693,0.498,0.609,0.404,0.698,0.769,0.787,0.747,0.431,0.453,0.538,0.502,0.564,0.693,0.791,0.516,0.485,0.662,0.413,0.551
Microstates+GB,0.576,20,0.609,0.396,0.667,0.493,0.613,0.813,0.907,0.462,0.507,0.422,0.498,0.436,0.449,0.631,0.658,0.507,0.520,0.720,0.627,0.582
VAR+shLDA,0.561,20,0.444,0.489,0.524,0.529,0.520,0.804,0.764,0.596,0.636,0.378,0.556,0.613,0.413,0.702,0.889,0.502,0.298,0.542,0.356,0.658


,OOD_AUROC,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20
method,,,,,,,,,,,,,,,,,,,,,,
EEGNet+TempDens,0.642,20,0.594,0.567,0.733,0.521,0.739,0.879,0.726,0.404,0.631,0.561,0.677,0.626,0.601,0.738,0.876,0.537,0.506,0.587,0.589,0.761
ShallowConvNet+kNN,0.636,20,0.592,0.598,0.694,0.477,0.727,0.809,0.738,0.416,0.787,0.488,0.487,0.609,0.616,0.752,0.793,0.534,0.531,0.704,0.601,0.770
FBCSP+IsolationForest,0.629,20,0.531,0.631,0.588,0.517,0.744,0.787,0.636,0.467,0.778,0.499,0.689,0.634,0.626,0.658,0.763,0.528,0.500,0.851,0.513,0.632
VAR+Mahalanobis,0.578,20,0.517,0.586,0.598,0.414,0.628,0.781,0.588,0.561,0.704,0.549,0.618,0.541,0.522,0.629,0.609,0.530,0.497,0.554,0.484,0.657
Log-bandpower+Mahalanobis,0.570,20,0.526,0.609,0.562,0.458,0.652,0.672,0.547,0.417,0.757,0.647,0.614,0.558,0.636,0.523,0.505,0.528,0.401,0.707,0.466,0.608
ACM+TS+OCSVM,0.557,20,0.434,0.506,0.499,0.440,0.663,0.739,0.495,0.495,0.791,0.609,0.651,0.544,0.535,0.592,0.581,0.537,0.412,0.681,0.327,0.603
Microstates+LOF,0.546,20,0.506,0.492,0.398,0.468,0.574,0.793,0.609,0.509,0.617,0.404,0.604,0.573,0.596,0.488,0.542,0.531,0.506,0.664,0.461,0.576


In [5]:
display_summary_by_dataset(
    experiment,
    mi_pipelines.keys(),
    ood_pipelines.keys(),
    experiment_specs=[
        ("MI BA", "mi", "MI_BA"),
        ("MI AUROC", "mi", "MI_AUROC"),
        ("OOD", "ood", "OOD_AUROC"),
    ],
)
